# Question 1 (25 points)
* Assume that the inputs `r` and `b` are the 4-dimensional vectors of sale prices and resource limits, respectively.
* The function must return the following sensitivity information (in order):
    1. The reduced cost associated with the optimal number of type~1 chips produced.
    2. The optimal dual variable associated with the constraint on total testing time.
    3. The minimum possible value by which `r[2]` can be perturbed so that the optimal solution remains unchanged.
    4. The maximum possible value by which `b[4]` can be perturbed so that the optimal solution remains unchanged.

## Note
* Do not change the name of the function or its signature (i.e., DO NOT replace `r` with `r::Vector` or `b` with `b::Vector`).
* Make sure that the function always returns its outputs in the order specified above.
    * Changing the order will make it impossible to automatically score your submission on Gradescope.

## Hints
* The model for Homework 9, Q1(a) is a **maximization** problem, for which the optimal reduced costs are always $\leq 0$.
* Optimal reduced costs can be directly obtained using the function `reduced_cost`. See [this link](https://jump.dev/JuMP.jl/stable/reference/solutions/#JuMP.reduced_cost).
* Optimal dual variable values can be directly obtained the function `shadow_price`. See [this link](https://jump.dev/JuMP.jl/stable/reference/solutions/#JuMP.shadow_price).
    - NOTE: You must name all constraints before you can use this function!
* Sensitivity information can be directly obtained using the function `lp_sensitivity_report`. See [this link](https://jump.dev/JuMP.jl/stable/tutorials/linear/lp_sensitivity/).
    - Read the sections on 'Variable sensitivity' and 'Constraint sensitivity' to see how you can extract the required information.

In [1]:
using JuMP, GLPK

function production_planning_sensitivity(r, b)
    
    # create an optimization model using the GLPK solver
    m = Model(GLPK.Optimizer)
    
    # declare variables
    @variable(m, x[1:4] >= 0)
    
    # objective function - max revenue
    @objective(m, Max, r'x) 
    
    # constraints
    @constraint(m, raw , x[1]+x[2]+x[3]+x[4] <= b[1])
    @constraint(m, etching , x[1]+x[2]+2*x[3]+2*x[4] <= b[2])
    @constraint(m, lamination , 2*x[1]+2*x[2]+3*x[3]+2*x[4] <= b[3])
    @constraint(m, testing , 2*x[1]+x[2]+3*x[3]+3*x[4] <= b[4])
    
    # solve model
    optimize!(m)
    
    if termination_status(m) != OPTIMAL
        optimal_solution = nothing
        @warn("The investment model was not solved correctly!")
    else  
        optimal_solution = value.(x)
        optimal_solution = round.(optimal_solution ; digits=2)
    end
    
    # reduced cost of x1
    cbar_1 = reduced_cost(x[1])
    
    # optimal dual var. for testing time
    opt_dual_test = round.(shadow_price(testing) ; digits=2)
    
    # sensitivity analysis
    sensitivity = lp_sensitivity_report(m)
    min_price1 = minimum(sensitivity[x[2]])
    extra_testing = maximum(sensitivity[testing])
    
    return cbar_1, opt_dual_test, min_price1, extra_testing
    
end

production_planning_sensitivity (generic function with 1 method)

In [2]:
r = [20,30,50,40]
b = [4000,6000,9000,7000]

production_planning_sensitivity(r, b)

(-15.0, 5.0, -3.3333333333333317, 1000.0)

# Question 2 (15 points)
* Assume that the inputs `r`, `a`, `s`, `h`, `d` are 20-dimensional vectors corresponding to each player's rebounding, assists, scoring, height, and defense ability, respectively.
* Assume that the inputs `r_mean`, `a_mean`, `s_mean`, `h_mean`, `d_mean` are floating point numbers corresponding to the overall team's mean required performance in each of the above categories.
* The function must return the optimal team composition as a single vector. For example, if the optimal team consists of players 1,2,...,12, then the function must return `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]`.

## Note
* Do not change the name of the function or its signature (i.e., DO NOT replace `r` with `r::Vector` etc.).
* Make sure that the function always returns a vector.

## Hints
* You can check if the optimal value of a binary variable `x` is equal to `1` simply by writing `if value(x) > 0.5`.
    - Because of floating point errors, the optimal value of a binary variable will be either very close to `0` or very close to `1`. Values that are very close to `0` should be treated as being equal to `0` and values that are very close to `1` should be treated as being equal to `1`. Therefore, we can simply check if the optimal value is smaller or greater than `0.5` to decide if it is equal to `0` or `1`, respectively.
* Although it is possible that the model can be infeasible for certain inputs, the data corresponding to all of the secret inputs are such that you always have a feasible solution (provided you have formulated your model correctly!).

In [3]:
using JuMP, GLPK

function solve_team_selection_problem(r, a, s, h, d, r_mean, a_mean, s_mean, h_mean, d_mean)
    
    # create an optimization model using the GLPK solver
    m = Model(GLPK.Optimizer)
    
    # declare binary variables
    @variable(m, x[1:20], Bin)
    
    # objective function - max scoring avg
    @objective(m, Max, s'x) 
    
    # constraints
    @constraint(m, selected_players, sum(x[i] for i in 1:20) == 12)
    @constraint(m, PM_players, sum(x[i] for i in 1:5) >= 3)
    @constraint(m, SG_players, sum(x[i] for i in 4:11) >= 4)
    @constraint(m, F_players, sum(x[i] for i in 9:16) >= 4)
    @constraint(m, C_players, sum(x[i] for i in 16:20) >= 3)
    @constraint(m, rmean , sum(x[i]*r[i] for i in 1:20)/12 >= r_mean)
    @constraint(m, amean , sum(x[i]*a[i] for i in 1:20)/12 >= a_mean)
    @constraint(m, smean , sum(x[i]*s[i] for i in 1:20)/12 >= s_mean)
    @constraint(m, hmean , sum(x[i]*h[i] for i in 1:20)/12 >= h_mean)
    @constraint(m, dmean , sum(x[i]*d[i] for i in 1:20)/12 >= d_mean)
    @constraint(m, p9_nop5, (x[5]+x[9]) <= 1)
    @constraint(m, p2_and_p19, x[2] == x[19])
    @constraint(m, same_team_players, (x[1]+x[7]+x[12]+x[16]) <= 3)
    
    # solve model
    optimize!(m)
    
    if termination_status(m) != OPTIMAL
        optimal_solution = nothing
        @warn("The investment model was not solved correctly!")
    else  
        optimal_solution = value.(x)
    
    end
    
    team = []
    for i in 1:20
        if value(x[i]) > 0.5
            [i]
            push!(team,i)
        end
    end
    
    return team

end

solve_team_selection_problem (generic function with 1 method)

In [4]:
r = rand(6:10,20)
a = rand(5:10,20)
s = rand(6:10,20)
h = [ 1.7,1.8 ,1.75 ,1.9 ,1.87 ,1.96 , 1.84,1.8 ,1.86 ,1.79, 1.91, 1.9, 1.88, 1.77, 1.89, 1.96, 1.91, 1.92,1.89 ,1.94 ]
d = rand(5:9,20)
r_mean = 8.5
a_mean = 7.5
s_mean = 8
h_mean = 1.85
d_mean = 7

solve_team_selection_problem(r, a, s, h, d, r_mean, a_mean, s_mean, h_mean, d_mean)

12-element Vector{Any}:
  1
  2
  4
  5
  8
 10
 14
 15
 16
 17
 19
 20